# Notebook 07 — Comparativo com Redes Neurais (TF-IDF vs Embeddings) e AutoML

Este notebook expande a análise adicionando Redes Neurais à comparação. O objetivo é testar se o *Reality Gap* (Visão 3) observado nos modelos lineares pode ser mitigado pelo uso de arquiteturas que capturam o contexto e relações não-lineares.

**NOVIDADE: Ajuste Fino Automático (Grid Search Extremo)**
Adicionamos testes com redes neurais gigantescas (até 10 camadas na LSTM e 1000 neurônios na MLP) para ver se o "tamanho do cérebro" consegue resolver a falta de dados.

*⚠️ AVISO CRÍTICO: Modelos com 10 camadas de LSTM ou 1000 neurônios demoram EXTREMAMENTE no processador (CPU). Reduzimos para 3 épocas os testes gigantes para o seu PC não derreter, mas ainda assim levará bastante tempo.*

In [1]:
# 0. SETUP: Detecção de Ambiente e Instalação
import os, sys, re, glob, shutil

IN_COLAB = 'google.colab' in sys.modules

try:
    import kagglehub
    import ipywidgets
    import torch
    import tqdm
except ImportError:
    print("Instalando dependências necessárias...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub", "ipywidgets", "torch", "tqdm", "-q", "--break-system-packages"])
    import torch
    print("✅ Instalação concluída!")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Para a Rede Simples (MLP) usando TF-IDF e Busca Automática
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

# Para a Rede Profunda (LSTM) usando PyTorch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

if IN_COLAB:
    print("Ambiente: Google Colab")
    REPO_URL = 'https://github.com/ROMAUSKI/tcc-analise-sentimento.git'
    if not os.path.exists('tcc-analise-sentimento'):
        import subprocess
        subprocess.check_call(["git", "clone", REPO_URL])
    os.chdir('tcc-analise-sentimento')
else:
    print("Ambiente: Local (VS Code)")
    if os.getcwd().endswith('src'):
        os.chdir('..')

sns.set_style('whitegrid')

Ambiente: Local (VS Code)


In [2]:
# 1. CONFIGURAÇÃO E CARREGAMENTO DE DADOS
NICHO = 'movies'
RANDOM_SEED = 42

CONFIG = {
    'movies': { 'raw_synth': 'dados/brutos/', 'real_file': 'src/dados_reais/utlc_movies.csv' },
    'apps': { 'raw_synth': 'dados/brutos_apps/', 'real_file': 'src/dados_reais/utlc_apps.csv' }
}

def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-zÀ-ÿ\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

arquivos = [f for f in glob.glob(os.path.join(CONFIG[NICHO]['raw_synth'], '*.csv')) if all(x not in f for x in ['metadata', 'dataset', 'synthetic'])]
lista = []
for f in arquivos:
    nome = os.path.basename(f).lower()
    classe = 'Positiva' if 'positive' in nome else ('Negativa' if 'negative' in nome else 'Neutra')
    df_temp = pd.read_csv(f, header=None, names=['frase'], usecols=[0])
    df_temp['classe'] = classe
    lista.append(df_temp)
df_sintetico = pd.concat(lista, ignore_index=True)
df_sintetico['frase_limpa'] = df_sintetico['frase'].apply(limpar_texto)
df_sintetico = df_sintetico.drop_duplicates(subset=['frase_limpa']).reset_index(drop=True)

path_real = CONFIG[NICHO]['real_file']
if not os.path.exists(path_real):
    d_path = kagglehub.dataset_download("fredericods/ptbr-sentiment-analysis-datasets")
    shutil.copy(os.path.join(d_path, os.path.basename(path_real)), path_real)
df_real_raw = pd.read_csv(path_real, nrows=100000)
def mapper(rating):
    if rating >= 4: return 'Positiva'
    elif rating <= 2: return 'Negativa'
    else: return 'Neutra'
df_real_raw['classe'] = df_real_raw['rating'].apply(mapper)
df_real = df_real_raw[['review_text', 'classe']].dropna()
df_real.columns = ['frase', 'classe']
df_real['frase_limpa'] = df_real['frase'].apply(limpar_texto)
df_real = df_real[df_real['frase_limpa'] != ""]
min_size = df_real['classe'].value_counts().min()
df_real_bal = df_real.groupby('classe').sample(n=min_size, random_state=RANDOM_SEED)

print(f"Dataset Sintético pronto: {len(df_sintetico)} amostras")
print(f"Dataset Real pronto: {len(df_real_bal)} amostras")

Dataset Sintético pronto: 1800 amostras
Dataset Real pronto: 27687 amostras


In [3]:
# 2. PREPARAÇÃO PARA REDES NEURAIS (ENCODING DE CLASSES)
class_map = {'Negativa': 0, 'Neutra': 1, 'Positiva': 2}
y_sintetico_num = df_sintetico['classe'].map(class_map).values
y_real_num = df_real_bal['classe'].map(class_map).values

R_tr, R_te, y_R_tr, y_R_te = train_test_split(df_real_bal['frase_limpa'], y_real_num, test_size=0.2, stratify=y_real_num, random_state=RANDOM_SEED)
S_tr, S_te, y_S_tr, y_S_te = train_test_split(df_sintetico['frase_limpa'], y_sintetico_num, test_size=0.2, stratify=y_sintetico_num, random_state=RANDOM_SEED)
X_V3_train, y_V3_train = df_sintetico['frase_limpa'], y_sintetico_num
X_V3_test, y_V3_test = df_real_bal['frase_limpa'], y_real_num

In [4]:
# 3. EXPERIMENTO A: MLP COM GRID SEARCH (BUSCA AUTOMÁTICA)

def run_mlp_automl(X_train, y_train, X_test, y_test, visao_nome):
    print(f"\n[AutoML MLP - {visao_nome}] Testando combinações extremas...")
    tfidf = TfidfVectorizer()
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)
    
    # --- A MALHA DE TESTES (GRID GIGANTE) ---
    # Testando redes rasas, médias, e uma rede monstruosa de (1000 neurônios na 1ª, 500 na 2ª)
    parametros_para_testar = {
        'hidden_layer_sizes': [(100,), (50, 50), (500,), (100, 100, 100), (1000, 500, 100)],
        'learning_rate_init': [0.001, 0.01]
    }
    
    mlp_base = MLPClassifier(max_iter=300, random_state=RANDOM_SEED, early_stopping=True)
    
    grid_search = GridSearchCV(mlp_base, parametros_para_testar, cv=3, n_jobs=-1, scoring='f1_weighted')
    grid_search.fit(X_train_tfidf, y_train)
    
    melhor_modelo = grid_search.best_estimator_
    print(f"  -> MELHOR COMBINAÇÃO ENCONTRADA: {grid_search.best_params_}")
    
    y_pred = melhor_modelo.predict(X_test_tfidf)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    print(f"  -> F1-Score do Melhor Modelo: {f1:.2%}")
    
    return {'Modelo': 'MLP (AutoML Extremo)', 'Visão': visao_nome, 'F1-Score': f1, 'Melhores Params': str(grid_search.best_params_)}

results_mlp = []
results_mlp.append(run_mlp_automl(R_tr, y_R_tr, R_te, y_R_te, "V1: Real -> Real"))
results_mlp.append(run_mlp_automl(S_tr, y_S_tr, S_te, y_S_te, "V2: Sint. -> Sint."))
results_mlp.append(run_mlp_automl(X_V3_train, y_V3_train, X_V3_test, y_V3_test, "V3: Sint. -> Real"))


[AutoML MLP - V1: Real -> Real] Testando combinações extremas...


OSError: [WinError 1450] Não existem recursos de sistema suficientes para concluir o serviço solicitado

In [ ]:
# 4. EXPERIMENTO B: DEEP LEARNING (LSTM) COM AJUSTE FINO EXTREMO
from tqdm.notebook import tqdm
import itertools

MAX_WORDS = 20000
MAX_LEN = 100

class SimpleTokenizer:
    def __init__(self, num_words):
        self.num_words = num_words
        self.word_index = {}
    def fit_on_texts(self, texts):
        counter = Counter()
        for text in texts: counter.update(text.split())
        common_words = counter.most_common(self.num_words - 2)
        self.word_index = {w: i+2 for i, (w, _) in enumerate(common_words)}
        self.word_index['<PAD>'] = 0
        self.word_index['<UNK>'] = 1
    def texts_to_sequences(self, texts):
        return [[self.word_index.get(w, 1) for w in text.split()] for text in texts]

def pad_sequences(sequences, maxlen):
    padded = np.zeros((len(sequences), maxlen), dtype=int)
    for i, seq in enumerate(sequences):
        length = min(len(seq), maxlen)
        if length > 0: padded[i, -length:] = seq[-length:]
    return padded

class LSTMClassifier(nn.Module):
    # NOVIDADE: O modelo agora aceita num_layers (Empilhar LSTMs uma em cima da outra)
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, text):
        embedded = self.embedding(text)
        _, (hidden, _) = self.lstm(embedded)
        # Pega a última camada (se tivermos 10 camadas, pegamos o pensamento da 10ª)
        return self.fc(hidden[-1,:,:])

def run_lstm_automl(X_train_text, y_train, X_test_text, y_test, visao_nome):
    print(f"\n[AutoML LSTM - {visao_nome}] Iniciando busca de hiperparâmetros (CUIDADO: PODE DEMORAR HORAS)...")
    
    tokenizer = SimpleTokenizer(num_words=MAX_WORDS)
    tokenizer.fit_on_texts(X_train_text)
    X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
    X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_tensor_x = torch.tensor(X_train_pad, dtype=torch.long)
    train_tensor_y = torch.tensor(y_train, dtype=torch.long)
    
    # --- A MALHA DE TESTES DA LSTM (NÍVEL LOUCURA) ---
    param_grid = {
        'batch_size': [64, 128],   
        'learning_rate': [0.001],
        'hidden_dim': [64, 256],     # Memória de 64 (Normal) vs 256 (Gigante)
        'num_layers': [1, 3, 10]     # 1 Camada (Normal), 3 (Deep), 10 (Extremamente Profunda)
    }
    
    keys, values = zip(*param_grid.items())
    combinacoes = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    melhor_f1 = 0.0
    melhor_modelo_params = {}
    
    for i, params in enumerate(combinacoes):
        print(f"  -> Testando Combinação {i+1}/{len(combinacoes)}: {params}")
        
        train_data = TensorDataset(train_tensor_x, train_tensor_y)
        train_loader = DataLoader(train_data, batch_size=params['batch_size'], shuffle=True)
        
        model = LSTMClassifier(MAX_WORDS, 128, params['hidden_dim'], 3, num_layers=params['num_layers']).to(device)
        optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        # Limitado a apenas 3 épocas. Se colocarmos mais, o teste com 10 camadas não terminaria hoje.
        for epoch in range(3):
            for texts, labels in train_loader:
                texts, labels = texts.to(device), labels.to(device)
                optimizer.zero_grad()
                predictions = model(texts)
                loss = criterion(predictions, labels)
                loss.backward()
                optimizer.step()
                
        model.eval()
        with torch.no_grad():
            X_test_tensor = torch.tensor(X_test_pad, dtype=torch.long).to(device)
            predictions = model(X_test_tensor)
            y_pred = torch.argmax(predictions, dim=1).cpu().numpy()
            
        _, _, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
        print(f"     Resultado: F1 = {f1:.2%}")
        
        if f1 > melhor_f1:
            melhor_f1 = f1
            melhor_modelo_params = params
            
    print(f"\n  🏆 VENCEDOR DA VISÃO: {melhor_modelo_params} com F1 = {melhor_f1:.2%}")
    return {'Modelo': 'LSTM (AutoML Extremo)', 'Visão': visao_nome, 'F1-Score': melhor_f1, 'Melhores Params': str(melhor_modelo_params)}

results_lstm = []
results_lstm.append(run_lstm_automl(R_tr, y_R_tr, R_te, y_R_te, "V1: Real -> Real"))
results_lstm.append(run_lstm_automl(S_tr, y_S_tr, S_te, y_S_te, "V2: Sint. -> Sint."))
results_lstm.append(run_lstm_automl(X_V3_train, y_V3_train, X_V3_test, y_V3_test, "V3: Sint. -> Real"))

In [ ]:
# 5. CONSOLIDAÇÃO E VISUALIZAÇÃO
df_nn = pd.DataFrame(results_mlp + results_lstm)

# Para ter a foto completa, vamos pegar só os resultados do SVM do Notebook 04 para servir de Baseline
baseline_data = [
    {'Modelo': 'Baseline SVM', 'Visão': 'V1: Real -> Real', 'F1-Score': 0.5452, 'Melhores Params': 'N/A'},
    {'Modelo': 'Baseline SVM', 'Visão': 'V2: Sint. -> Sint.', 'F1-Score': 0.8915, 'Melhores Params': 'N/A'},
    {'Modelo': 'Baseline SVM', 'Visão': 'V3: Sint. -> Real', 'F1-Score': 0.4024, 'Melhores Params': 'N/A'}
]
df_base = pd.DataFrame(baseline_data)

df_final = pd.concat([df_base, df_nn], ignore_index=True)
df_final.to_csv('resultados/metricas_redes_neurais_automl.csv', index=False)

print("\n--- TABELA COMPARATIVA (COM MELHORES PARÂMETROS DESCOBERTOS) ---")
display(df_final[['Visão', 'Modelo', 'F1-Score', 'Melhores Params']])

plt.figure(figsize=(12, 6))
sns.barplot(data=df_final, x='Visão', y='F1-Score', hue='Modelo', palette='Set2')
plt.title('Comparativo: O Impacto do Ajuste Fino (AutoML Extremo) nas Redes Neurais')
plt.ylim(0, 1.0)
plt.savefig('resultados/grafico_redes_neurais_automl.png')
plt.show()